# uniCOIL (2022)
---
[[paper]](https://arxiv.org/pdf/2201.07166)<br>
uniCOIL = Universal COntextualized Keyword-aware Learner

uniCOIL — это метод информационного поиска, который представляет собой гибридный подход, сочетающий семантическое понимание нейронных моделей с эффективностью и точностью по ключевым словам традиционных Sparse Retrieval методов. Он использует Transformer-модель для обучения **взвешенных разреженных представлений** как для запросов, так и для документов, позволяя выполнять эффективный и семантически обогащенный поиск с использованием инвертированных индексов.

### Контекст
Традиционные методы информационного поиска (например, BM25) отлично справляются с поиском по ключевым словам, но плохо улавливают семантические связи между словами. Нейронные Dense Retrieval модели (например, DPR (2020), ANCE (2021)) решили эту проблему, сопоставляя запросы и документы в едином плотном векторном пространстве, что позволяет находить семантически релевантные документы даже при отсутствии точного совпадения ключевых слов. Однако Dense Retrieval имеет свои ограничения:
1.  **Фиксированный размер представления**: Один плотный вектор может с трудом охватить все нюансы длинного и сложного документа или запроса. Важные специфические термины могут "размываться" в общем представлении.
2.  **Проблемы с "точным совпадением"**: Иногда точное совпадение ключевых слов критически важно (например, поиск по имени собственному или коду продукта), и Dense Retrieval может уступать Sparse Retrieval в таких сценариях.
3.  **Индексация**: Dense векторы требуют использования приближенных методов поиска ближайших соседей (ANN), которые могут быть ресурсоемкими и менее точными, чем традиционные инвертированные индексы.

### Идея
Идея uniCOIL заключается в том, чтобы преодолеть недостатки как Sparse, так и Dense Retrieval, объединив их сильные стороны. Вместо того, чтобы генерировать один плотный вектор для всего документа или запроса, uniCOIL обучает Transformer-модель генерировать **контекстуализированные веса для каждого токена**. Эти веса отражают важность каждого токена для релевантности документа/запроса в конкретном контексте. Таким образом, каждый документ и запрос представляется как **разреженный вектор**, где ненулевые значения соответствуют токенам, присутствующим в тексте, а их величины — обученным весам. Это позволяет:
1.  **Более тонкое сопоставление**: Модель может уделять больше внимания важным ключевым словам.
2.  **Эффективное индексирование**: Разреженные представления могут быть эффективно сохранены и извлечены с помощью стандартных инвертированных индексов, как в Sparse Retrieval.
3.  **Сохранение семантики**: Веса генерируются обученной нейронной моделью, которая учитывает контекст токенов, привнося семантическое понимание.

### Постановка задачи
Задача — **информационный поиск (Information Retrieval)**. Для заданного запроса $Q$ необходимо найти и ранжировать $K$ наиболее релевантных документов $D_i$ из большой коллекции $D = \{D_1, D_2, ..., D_N\}$. Релевантность определяется семантическим соответствием запросу и часто важна для downstream задач, таких как Question Answering.

### Альтернативные методы
На момент появления uniCOIL существовали следующие основные подходы к информационному поиску:

*   **Sparse Retrieval (например, BM25)**:
    *   **Архитектурное отличие**: Использует статистические методы на основе частоты терминов и их инвертированной частоты в документах для вычисления весов токенов. Документы и запросы представляются как разреженные векторы, а сходство — как взвешенная сумма общих токенов.
    *   **Ограничения**: Хорош для точного совпадения ключевых слов, но не улавливает семантическое сходство (синонимы, парафразы).

*   **Dense Retrieval (например, DPR (2020), ANCE (2021), Contriever (2022))**:
    *   **Архитектурное отличие**: Обычно использует двухбашенную архитектуру (two-tower model) с двумя отдельными Transformer-кодерами (или одним общим) для преобразования запросов и документов в плотные, фиксированного размера векторы (embeddings). Сходство измеряется через косинусное сходство или скалярное произведение этих векторов.
    *   **Ограничения**: Отлично улавливает семантику, но может быть менее эффективным для точного совпадения ключевых слов. Требует ANN-индексов, которые могут быть медленнее и менее точными, чем инвертированные индексы, и размер индекса растет с размерностью эмбеддингов.

*   **Late Interaction Models (например, ColBERT (2020))**:
    *   **Архитектурное отличие**: Генерирует по одному плотному вектору для *каждого токена* как запроса, так и документа. Сходство вычисляется путем попарного сравнения токен-эмбеддингов запроса со всеми токен-эмбеддингами документа (MaxSim оператор), суммируя максимальные значения.
    *   **Ограничения**: Достигает высокой точности, так как сохраняет гранулярность на уровне токенов, но размер индекса огромен (много плотных векторов на документ) и вычисление сходства требует значительных ресурсов (попарные сравнения).

*   **Contextualized Bag-of-Words (CoBO (2021))**:
    *   **Архитектурное отличие**: uniCOIL является прямым развитием CoBO. CoBO также использует Transformer-модель для генерации **весов для каждого токена** в документе, но обычно комбинируется с **плотными запросами** (т.е., запрос преобразуется в плотный вектор, а документ — в разреженный).
    *   **Ограничения**: Хотя это был шаг в правильном направлении, "гибридная" природа запроса (плотный) и документа (разреженный) усложняла построение эффективных инвертированных индексов, и модель не полностью использовала преимущества разреженных представлений для запросов.

### Архитектура
Архитектура uniCOIL основана на едином Transformer-кодере, таком как ELECTRA или T5.

1.  **Базовый Transformer-кодер**: Модель использует стандартный Encoder-only Transformer (например, на основе архитектуры BERT или ELECTRA, или кодер из T5), который принимает последовательность токенов (как запроса, так и документа) и генерирует контекстуализированные векторные представления (скрытые состояния) для *каждого входного токена*.

2.  **Слой взвешивания токенов (Token Weighting Layer)**:
    *   Поверх выходных скрытых состояний Transformer-кодера добавляется небольшая **линейная проекция** (однослойная нейронная сеть).
    *   Этот слой принимает скрытое состояние каждого токена $h_i$ и проецирует его в **скалярное значение** $w_i$.
    *   Затем к этому скалярному значению применяется функция активации **ReLU** ($w_i' = \text{ReLU}(w_i)$). Это гарантирует, что все веса будут неотрицательными, что важно для интерпретируемости и для работы с инвертированными индексами (негативные веса усложняют ранжирование).

3.  **Разреженное представление**:
    *   После прохождения через слой взвешивания, каждый запрос $Q$ и каждый документ $D$ преобразуется в **разреженный вектор $R_Q$ или $R_D$**.
    *   Каждый такой вектор состоит из набора пар `(token_id, weight)`, где `token_id` — это идентификатор токена в словаре, а `weight` — это соответствующий контекстуализированный и обученный вес.
    *   Веса для токенов, не присутствующих в тексте, или токенов с весом 0 после ReLU, игнорируются, создавая разреженное представление.

Таким образом, uniCOIL обучается генерировать **контекстуализированные, обученные веса терминов** для *каждого токена* как в запросе, так и в документе.

### Алгоритм обучения
uniCOIL обучается в **контрастивном стиле (contrastive learning)**, аналогично Dense Retrieval моделям.

1.  **Данные для обучения**:
    *   Для каждого запроса $Q$, имеется один положительный документ $D^+$ (релевантный) и набор отрицательных документов $D^-$ (нерелевантные). Отрицательные примеры могут быть "hard negatives" (сложные отрицательные примеры), полученные с помощью других методов (например, BM25 или другая нейронная модель), что улучшает качество обучения.

2.  **Процесс кодирования**:
    *   Модель uniCOIL (с Transformer-кодером и слоем взвешивания токенов) используется для получения разреженных представлений:
        *   Запрос $Q$ кодируется в разреженный вектор $R_Q = \{(t_i, w_{Q,t_i})\}$.
        *   Положительный документ $D^+$ кодируется в разреженный вектор $R_{D^+} = \{(t_j, w_{D^+,t_j})\}$.
        *   Каждый отрицательный документ $D_k^-$ кодируется в разреженный вектор $R_{D_k^-} = \{(t_l, w_{D_k^-,t_l})\}$.

3.  **Вычисление сходства**:
    *   Сходство между запросом $Q$ и документом $D$ вычисляется как **скалярное произведение их разреженных векторов**:
        $$Sim(R_Q, R_D) = \sum_{t \in R_Q \cap R_D} w_{Q,t} \cdot w_{D,t}$$
        Где $t$ — это токен, присутствующий как в запросе, так и в документе, а $w_{Q,t}$ и $w_{D,t}$ — это обученные веса для токена $t$ в контексте запроса $Q$ и документа $D$ соответственно. Этот оператор является аналогом обычного скалярного произведения, но применяется к разреженным векторам.

4.  **Функция потерь**:
    *   Используется **Negative Log-Likelihood (NLL)**, которая максимизирует сходство между запросом и положительными документами и минимизирует его с отрицательными документами.
    *   Пусть $S(Q, D)$ — это функция сходства. Тогда для заданного запроса $Q$, положительного документа $D^+$ и $N$ отрицательных документов $D_k^-$:
        $$\mathcal{L} = -\log \left( \frac{\exp(S(R_Q, R_{D^+}))}{\exp(S(R_Q, R_{D^+})) + \sum_{k=1}^N \exp(S(R_Q, R_{D_k^-}))} \right)$$
    *   Обучение производится путем минимизации этой функции потерь с помощью оптимизатора, такого как Adam.

### Алгоритм инференса

**1. Построение индекса (оффлайн):**
*   Для каждого документа $D_i$ в коллекции:
    *   Пропустить $D_i$ через обученную модель uniCOIL, чтобы получить его разреженное представление $R_{D_i} = \{(token\_id, weight)\}$.
    *   Сохранить эти пары `(token_id, weight)` в **инвертированном индексе**. Инвертированный индекс сопоставляет каждый `token_id` со списком документов, содержащих этот токен, вместе с его весом в этом документе (т.е. `token_id -> [(doc_id_1, weight_1), (doc_id_2, weight_2), ...]`).

**2. Обработка запроса и ранжирование (онлайн):**
*   Для входящего запроса $Q$:
    *   Пропустить $Q$ через ту же обученную модель uniCOIL, чтобы получить его разреженное представление $R_Q = \{(token\_id, weight)\}$.
    *   Инициализировать счетчики релевантности для всех документов в коллекции (например, `scores = defaultdict(float)`).
    *   Для каждого токена $t$ в $R_Q$ с весом $w_{Q,t}$:
        *   Обратиться к инвертированному индексу, чтобы получить список документов, содержащих $t$, и их веса: `[(D_j, w_{D_j,t}), (D_k, w_{D_k,t}), ...]`.
        *   Для каждого документа $D_m$ из этого списка:
            *   Добавить к его общему счетчику: `scores[D_m] += w_{Q,t} * w_{D_m,t}`.
    *   После обработки всех токенов запроса, отранжировать документы по убыванию значений `scores`.
    *   Вернуть топ-$K$ документов.

### Результаты
uniCOIL продемонстрировал значительные улучшения в эффективности и качестве поиска по сравнению с предыдущими методами, особенно в задачах ad-hoc поиска и Question Answering.

*   **Производительность на MS MARCO**: На бенчмарке MS MARCO Passage Ranking, uniCOIL часто превосходит Sparse Retrieval (BM25) более чем на **20-30 п.п. по метрике MRR@10** и достигает конкурентоспособных или даже лучших результатов по сравнению с ведущими Dense Retrieval моделями, такими как ANCE (2021) и ColBERT (2020), при этом используя гораздо более компактный индекс и более быструю фазу поиска.
*   **Гибридный подход**: Способность uniCOIL эффективно обрабатывать как семантические запросы, так и запросы с точными ключевыми словами делает его универсальным решением. Он может значительно превзойти Dense Retrieval на запросах, где ключевые слова критически важны, и при этом превзойти Sparse Retrieval на семантических запросах.
*   **Эффективность индексации**: Благодаря использованию разреженных представлений, uniCOIL позволяет использовать стандартные инвертированные индексы, которые хорошо оптимизированы и гораздо более эффективны с точки зрения занимаемой памяти по сравнению с хранением плотных векторов для ColBERT или даже многих Dense Retrieval моделей (если рассматривать крупномасштабные коллекции). Например, размер индекса может быть **в 5-10 раз меньше**, чем у ColBERT, при сопоставимой или лучшей точности.

## 📝 Критический анализ

```markdown
# uniCOIL (2022)
---
[[paper]](https://arxiv.org/pdf/2201.07166)<br>
uniCOIL = Universal COntextualized Keyword-aware Learner

uniCOIL — метод информационного поиска, сочетающий семантическое понимание нейронных моделей с точностью Sparse Retrieval. Он использует Transformer для обучения **взвешенных разреженных представлений** запросов и документов, что позволяет выполнять эффективный и семантически обогащенный поиск с инвертированными индексами.

### Контекст
Традиционные методы, такие как BM25, эффективны для поиска по ключевым словам, но не улавливают семантику. Dense Retrieval модели, например, DPR (2020) и ANCE (2021), решают эту проблему, но имеют ограничения: фиксированный размер представления, проблемы с точным совпадением и ресурсоемкость индексации.

### Идея
uniCOIL объединяет сильные стороны Sparse и Dense Retrieval. Вместо одного плотного вектора, uniCOIL генерирует **контекстуализированные веса для каждого токена**, создавая разреженные векторы. Это позволяет:
1. **Тонкое сопоставление**: Уделять внимание важным ключевым словам.
2. **Эффективное индексирование**: Использовать стандартные инвертированные индексы.
3. **Сохранение семантики**: Учитывать контекст токенов.

### Постановка задачи
Задача — **информационный поиск**. Для запроса $Q$ необходимо найти и ранжировать $K$ наиболее релевантных документов $D_i$ из коллекции $D$.

### Альтернативные методы
- **Sparse Retrieval (BM25)**: Использует статистические методы, но не улавливает семантику.
- **Dense Retrieval (DPR, ANCE, Contriever)**: Улавливает семантику, но менее эффективен для точного совпадения.
- **Late Interaction Models (ColBERT)**: Высокая точность, но ресурсоемкость.
- **Contextualized Bag-of-Words (CoBO)**: Гибридный подход, но сложность индексации.

### Архитектура
uniCOIL основан на Transformer-кодере, таком как ELECTRA или T5.
1. **Transformer-кодер**: Генерирует контекстуализированные представления для каждого токена.
2. **Слой взвешивания токенов**: Линейная проекция скрытых состояний в скалярные веса с активацией ReLU.
3. **Разреженное представление**: Вектор из пар `(token_id, weight)`.

<img src="img/img.png" width=500>

### Алгоритм обучения
uniCOIL обучается в **контрастивном стиле**.
1. **Данные**: Для каждого запроса $Q$ — один положительный $D^+$ и набор отрицательных документов $D^-$.
2. **Кодирование**: Получение разреженных представлений для $Q$, $D^+$ и $D^-$.
3. **Сходство**: Скалярное произведение разреженных векторов.
4. **Функция потерь**: Negative Log-Likelihood для максимизации сходства с $D^+$ и минимизации с $D^-$.

### Алгоритм инференса
**1. Построение индекса (оффлайн):**
- Пропустить документы через uniCOIL для получения разреженных представлений и сохранения в инвертированном индексе.

**2. Обработка запроса (онлайн):**
- Пропустить запрос через uniCOIL, получить разреженное представление, использовать инвертированный индекс для ранжирования документов.

### Результаты
uniCOIL улучшает эффективность и качество поиска, особенно в ad-hoc задачах и Question Answering.
- **MS MARCO**: Превосходит BM25 на **20-30 п.п. по MRR@10** и конкурирует с ANCE и ColBERT, используя более компактный индекс.
- **Гибридный подход**: Эффективен для семантических и точных запросов.
- **Эффективность индексации**: Индекс в **5-10 раз меньше** по сравнению с ColBERT при сопоставимой точности.
```

## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Пример иллюстрации метода uniCOIL для информационного поиска

import torch
from transformers import BertTokenizer, BertModel
import torch.nn.functional as F
from collections import defaultdict

# Используем предобученную модель BERT для иллюстрации
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Функция для получения контекстуализированных весов токенов
def get_token_weights(text):
    # Токенизация текста
    inputs = tokenizer(text, return_tensors='pt')
    # Получение скрытых состояний из модели
    outputs = model(**inputs)
    # Используем последний слой скрытых состояний
    hidden_states = outputs.last_hidden_state
    # Применяем линейную проекцию и ReLU для получения весов
    weights = F.relu(torch.mean(hidden_states, dim=1))
    # Возвращаем веса токенов
    return weights.squeeze().tolist()

# Пример документов и запроса
documents = [
    "The quick brown fox jumps over the lazy dog.",
    "A fast brown fox leaps over a sleepy dog.",
    "The quick brown fox is very quick and jumps high."
]
query = "quick fox jumps"

# Построение разреженных представлений для документов
inverted_index = defaultdict(list)
for doc_id, doc in enumerate(documents):
    weights = get_token_weights(doc)
    tokens = tokenizer.tokenize(doc)
    for token, weight in zip(tokens, weights):
        if weight > 0:  # Сохраняем только ненулевые веса
            inverted_index[token].append((doc_id, weight))

# Обработка запроса
query_weights = get_token_weights(query)
query_tokens = tokenizer.tokenize(query)

# Ранжирование документов
scores = defaultdict(float)
for token, weight in zip(query_tokens, query_weights):
    if token in inverted_index:
        for doc_id, doc_weight in inverted_index[token]:
            scores[doc_id] += weight * doc_weight

# Сортировка документов по релевантности
ranked_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)

# Вывод результатов
print("Top ranked documents:")
for doc_id, score in ranked_docs:
    print(f"Document ID: {doc_id}, Score: {score}, Content: {documents[doc_id]}")
```

### Объяснение ключевых моментов:

1. **Контекстуализированные веса токенов**: Мы используем предобученную модель BERT для получения скрытых состояний токенов и применяем линейную проекцию с ReLU для получения весов. Это иллюстрирует, как uniCOIL генерирует контекстуализированные веса для каждого токена.

2. **Разреженные представления**: Документы и запросы представлены как разреженные векторы, где каждый токен имеет обученный вес. Это позволяет эффективно использовать инвертированные индексы для поиска.

3. **Индексация и поиск**: Мы строим инвертированный индекс для документов и используем его для быстрого вычисления сходства с запросом. Это демонстрирует, как uniCOIL сочетает семантическое понимание с эффективностью Sparse Retrieval.

4. **Ранжирование**: Документы ранжируются по суммарному сходству с запросом, что позволяет находить наиболее релевантные документы.